# Shoealls — Synapse CSV 학습 (Google Colab)

**준비:**
- 런타임 → 런타임 유형 변경 → **T4 GPU** 선택
- 왼쪽 자물쇠(🔑) → Secrets → `GITHUB_TOKEN` 추가

**데이터 폴더 ID:** `10PW5sNFn5UQvtesV9zGhdvmUHRB-D7i9`

In [ ]:
import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

In [ ]:
# Drive 마운트 + Google 인증
from google.colab import drive, auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload
import io, os

drive.mount('/content/drive')
auth.authenticate_user()
svc = build('drive', 'v3', cache_discovery=False)
print('인증 완료')

In [ ]:
# Drive API로 CSV 파일 목록 가져오기
FOLDER_ID = '10PW5sNFn5UQvtesV9zGhdvmUHRB-D7i9'
DATA_DIR  = '/content/synapse_data'
os.makedirs(DATA_DIR, exist_ok=True)

def list_drive_files(folder_id, mime=None):
    files, token = [], None
    q = f"'{folder_id}' in parents and trashed=false"
    if mime: q += f" and mimeType='{mime}'"
    while True:
        resp = svc.files().list(
            q=q, fields='nextPageToken, files(id, name, mimeType)',
            pageSize=200, pageToken=token
        ).execute()
        files += resp.get('files', [])
        token = resp.get('nextPageToken')
        if not token: break
    return files

all_files = list_drive_files(FOLDER_ID)
csv_files = [f for f in all_files if f['name'].lower().endswith('.csv')]
subdirs   = [f for f in all_files if f['mimeType'] == 'application/vnd.google-apps.folder']

print(f'폴더 내 CSV: {len(csv_files)}개  |  하위폴더: {len(subdirs)}개')
print("
파일 목록 (최대 20개):")
for f in csv_files[:20]:
    print(f'  {f["name"]}')
if subdirs:
    print("
하위폴더:")
    for d in subdirs:
        print(f'  [{d["name"]}] id={d["id"]}')


In [ ]:
# 샘플 CSV 1개 다운로드 → 컬럼/데이터 구조 확인
import pandas as pd, io as _io

if csv_files:
    sample = csv_files[0]
    req = svc.files().get_media(fileId=sample["id"])
    buf = _io.BytesIO()
    from googleapiclient.http import MediaIoBaseDownload
    dl = MediaIoBaseDownload(buf, req, chunksize=2*1024*1024)
    done = False
    while not done: _, done = dl.next_chunk()
    buf.seek(0)
    sample_df = pd.read_csv(buf, low_memory=False)

    print(f"파일명: {sample['name']}")
    print(f"행 수: {len(sample_df):,}  |  컬럼 수: {len(sample_df.columns)}")
    print("
컬럼 목록:")
    for c in sample_df.columns:
        print(f'  {c} ({sample_df[c].dtype})')
    print("
샘플 데이터 (3행):")
    display(sample_df.head(3))
else:
    print("CSV 파일이 없습니다. 폴더 ID나 공유 설정을 확인하세요.")


In [ ]:
# 파일 전체 다운로드 (이미 있는 파일은 스킵)
# ※ 샘플 프리뷰 후 컬럼 구조에 맞게 아래 라벨 매핑을 수정하세요
def get_label(name):
    n = name.lower()
    if 'control' in n or 'hc' in n or 'normal' in n: return 0
    if 'pd' in n or 'patient' in n or 'parkinson' in n: return 1
    return None  # None → 스킵

def download_file(file_id, dest_path):
    req = svc.files().get_media(fileId=file_id)
    buf = _io.BytesIO()
    dl  = MediaIoBaseDownload(buf, req, chunksize=4*1024*1024)
    done = False
    while not done: _, done = dl.next_chunk()
    with open(dest_path, 'wb') as f:
        f.write(buf.getvalue())

target   = [(f['id'], f['name'], get_label(f['name'])) for f in csv_files
            if get_label(f['name']) is not None]
existing = set(os.listdir(DATA_DIR))
to_dl    = [(fid, name, lbl) for fid, name, lbl in target if name not in existing]
print(f'라벨 매핑: {len(target)}개  |  신규 다운로드: {len(to_dl)}개  |  이미 있음: {len(target)-len(to_dl)}개')

for i, (fid, name, lbl) in enumerate(to_dl, 1):
    download_file(fid, os.path.join(DATA_DIR, name))
    if i % 10 == 0 or i == len(to_dl):
        print(f'  {i}/{len(to_dl)} 완료')

labeled_files = [(os.path.join(DATA_DIR, name), lbl) for _, name, lbl in target]
print(f'
총 {len(labeled_files)}개 파일 준비 완료')


In [ ]:
# 레포 클론 or 업데이트
from google.colab import userdata
import sys, subprocess, os

try:
    token = userdata.get('GITHUB_TOKEN')
except:
    token = input('GitHub Token: ').strip()

BRANCH = 'claude/review-github-code-0rvAI'
REPO_URL = f'https://{token}@github.com/jg-shoealls/Shoealls.git'

if not os.path.exists('/content/Shoealls'):
    os.system(f'git clone -b {BRANCH} {REPO_URL} /content/Shoealls 2>&1 | tail -2')
else:
    os.system(f'git -C /content/Shoealls fetch origin {BRANCH} 2>&1 | tail -1')
    os.system(f'git -C /content/Shoealls reset --hard origin/{BRANCH} 2>&1 | tail -1')

sys.path.insert(0, '/content/Shoealls')
os.chdir('/content/Shoealls')
subprocess.run(['pip', 'install', '-q', 'pandas', 'scipy', 'scikit-learn', 'pyyaml', 'matplotlib'], check=True)

for k in list(sys.modules):
    if 'src.models' in k:
        del sys.modules[k]

print('준비 완료')

In [ ]:
# 모델 파일을 mag_baro 버전으로 강제 교체
import subprocess, sys

_fetch = subprocess.run(
    ['git', '-C', '/content/Shoealls', 'fetch', 'origin', 'claude/review-github-code-0rvAI'],
    capture_output=True, text=True
)
_co = subprocess.run(
    ['git', '-C', '/content/Shoealls', 'checkout', 'FETCH_HEAD', '--',
     'src/models/multimodal_gait_net.py',
     'src/models/encoders.py',
     'src/models/fusion.py',
     'configs/default.yaml'],
    capture_output=True, text=True
)
print(_fetch.stderr.strip() or 'fetch OK')
print(_co.stdout.strip() or _co.stderr.strip() or 'checkout OK')

# 이전에 로드된 모듈 캐시 제거
for k in [k for k in sys.modules if k.startswith('src.models')]:
    del sys.modules[k]

print('모델 파일 교체 완료')

In [ ]:
# 데이터셋 구성
import pandas as pd
import numpy as np
from torch.utils.data import Dataset, DataLoader, random_split
from src.data.preprocessing import preprocess_imu, preprocess_pressure

SEQ_LEN  = 128
GRID     = (16, 8)
IMU_COLS = [
    'LowerBack_Acc_X', 'LowerBack_Acc_Y', 'LowerBack_Acc_Z',
    'LowerBack_Gyr_X', 'LowerBack_Gyr_Y', 'LowerBack_Gyr_Z'
]

def parse_csv(path):
    try:
        df = pd.read_csv(path, low_memory=False)
        if not all(c in df.columns for c in IMU_COLS):
            return None

        # IMU: (T,6) → preprocess → (6, SEQ_LEN)
        imu_raw = df[IMU_COLS].dropna().values.astype(np.float32)
        if len(imu_raw) < 10:
            return None
        imu = preprocess_imu(imu_raw, SEQ_LEN)          # (6, SEQ_LEN)

        # 족저압: L/R 스칼라 → (T,16,8) → preprocess → (SEQ_LEN,1,16,8)
        T = len(df)
        pres_l = df['L Foot Pressure'].values.astype(np.float32) if 'L Foot Pressure' in df.columns else np.zeros(T)
        pres_r = df['R Foot Pressure'].values.astype(np.float32) if 'R Foot Pressure' in df.columns else np.zeros(T)
        pres_raw = np.zeros((T, 16, 8), dtype=np.float32)
        pres_raw[:, :8, :] = pres_l[:, None, None] / 255.0
        pres_raw[:, 8:, :] = pres_r[:, None, None] / 255.0
        pressure = preprocess_pressure(pres_raw, SEQ_LEN, GRID)  # (SEQ_LEN,1,16,8)

        # mag_baro: WearGait에 없으므로 0으로 채움 → (5, SEQ_LEN)
        mag_baro = np.zeros((5, SEQ_LEN), dtype=np.float32)

        return imu, pressure, mag_baro
    except Exception as e:
        return None

class WearGaitDataset(Dataset):
    def __init__(self, file_label_pairs):
        self.samples = []
        skip = 0
        for path, label in file_label_pairs:
            r = parse_csv(path)
            if r is None:
                skip += 1; continue
            imu, pressure, mag_baro = r
            self.samples.append((imu, pressure, mag_baro, label))
        print(f'로드: {len(self.samples)}개 (스킵: {skip}개)')

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        imu, pres, mb, lbl = self.samples[idx]
        return {
            'imu':      torch.from_numpy(imu),       # (6, 128)
            'pressure': torch.from_numpy(pres),      # (128, 1, 16, 8)
            'mag_baro': torch.from_numpy(mb),        # (5, 128)
            'label':    torch.tensor(lbl, dtype=torch.long)
        }

dataset = WearGaitDataset(labeled_files)
print(f'최종 샘플: {len(dataset)}개')

In [ ]:
# 설정 + 모델 + DataLoader
import torch.nn as nn, yaml

CONFIG_PATH = '/content/Shoealls/configs/default.yaml'
if os.path.exists(CONFIG_PATH):
    with open(CONFIG_PATH) as f:
        config = yaml.safe_load(f)
else:
    config = {
        'data': {
            'sequence_length': SEQ_LEN, 'imu_channels': 6,
            'pressure_grid_size': [16, 8], 'num_classes': 4
        },
        'model': {
            'fusion': {'embed_dim': 128, 'num_heads': 4, 'ff_dim': 256, 'num_layers': 2, 'dropout': 0.1},
            'imu_encoder': {'dropout': 0.1},
            'pressure_encoder': {'dropout': 0.1},
            'mag_baro_encoder': {'dropout': 0.1},
            'classifier': {'hidden_dims': [128], 'dropout': 0.1}
        },
        'training': {'batch_size': 16, 'epochs': 30, 'learning_rate': 1e-3, 'weight_decay': 1e-4}
    }
config['data']['num_classes'] = 4
if 'mag_baro_encoder' not in config.get('model', {}):
    config['model']['mag_baro_encoder'] = {'dropout': 0.1}

total   = len(dataset)
train_n = int(total * 0.7)
val_n   = int(total * 0.15)
train_ds, val_ds, test_ds = random_split(
    dataset, [train_n, val_n, total - train_n - val_n],
    generator=torch.Generator().manual_seed(42)
)
print(f'학습: {len(train_ds)} / 검증: {len(val_ds)} / 테스트: {len(test_ds)}')

BS = config['training']['batch_size']
train_loader = DataLoader(train_ds, batch_size=BS, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BS, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BS, shuffle=False, num_workers=2, pin_memory=True)

from src.models.multimodal_gait_net import MultimodalGaitNet
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model  = MultimodalGaitNet(config).to(device)
print(f'Device: {device} | 파라미터: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
# 학습
import time

EPOCHS   = config['training']['epochs']
SAVE_DIR = '/content/drive/MyDrive/WearGait-PD/checkpoints'
os.makedirs(SAVE_DIR, exist_ok=True)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(),
    lr=config['training']['learning_rate'], weight_decay=config['training']['weight_decay'])
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
best_val_acc = 0.0

def run_epoch(loader, train=True):
    model.train(train)
    total_loss, correct, total = 0, 0, 0
    with torch.set_grad_enabled(train):
        for batch in loader:
            imu  = batch['imu'].to(device)
            pres = batch['pressure'].to(device)
            mb   = batch['mag_baro'].to(device)
            lbl  = batch['label'].to(device)
            out  = model({'imu': imu, 'pressure': pres, 'mag_baro': mb})
            logits = out if isinstance(out, torch.Tensor) else out['logits']
            loss = criterion(logits, lbl)
            if train:
                optimizer.zero_grad(); loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
            total_loss += loss.item() * len(lbl)
            correct    += (logits.argmax(1) == lbl).sum().item()
            total      += len(lbl)
    return total_loss / total, correct / total

print('Epoch | TrLoss  | TrAcc  | VlLoss  | VlAcc  | Time')
print('-' * 52)
for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    tr_loss, tr_acc = run_epoch(train_loader, True)
    vl_loss, vl_acc = run_epoch(val_loader,   False)
    scheduler.step()
    history['train_loss'].append(tr_loss); history['val_loss'].append(vl_loss)
    history['train_acc'].append(tr_acc);  history['val_acc'].append(vl_acc)
    flag = ' *' if vl_acc > best_val_acc else ''
    print(f'{epoch:5d} | {tr_loss:.4f}  | {tr_acc:.4f} | {vl_loss:.4f}  | {vl_acc:.4f} | {time.time()-t0:.1f}s{flag}')
    if vl_acc > best_val_acc:
        best_val_acc = vl_acc
        torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(),
                    'config': config, 'val_accuracy': best_val_acc, 'model_type': 'basic'},
                   os.path.join(SAVE_DIR, 'best_model.pt'))

print(f'\n완료! 최고 Val Acc: {best_val_acc:.4f}')

In [ ]:
# 테스트 평가 + 종합 결과 리포트
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

# ── 최적 모델 로드
checkpoint = torch.load(os.path.join(SAVE_DIR, "best_model.pt"))
model.load_state_dict(checkpoint["model_state_dict"])
test_loss, test_acc = run_epoch(test_loader, False)
print(f"테스트 정확도: {test_acc:.4f} ({test_acc*100:.1f}%)")

# ── 예측값 수집
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for batch in test_loader:
        out = model({"imu": batch["imu"].to(device),
                     "pressure": batch["pressure"].to(device),
                     "mag_baro": batch["mag_baro"].to(device)})
        logits = out if isinstance(out, torch.Tensor) else out["logits"]
        all_preds.extend(logits.argmax(1).cpu().numpy())
        all_labels.extend(batch["label"].numpy())
all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)

# ── 그래프 레이아웃
fig = plt.figure(figsize=(18, 12))
fig.suptitle(f"Shoealls Gait AI — 학습 결과 리포트
"
             f"Test Acc: {test_acc*100:.1f}%  |  Best Val Acc: {best_val_acc*100:.1f}%  |  Epochs: {EPOCHS}",
             fontsize=14, fontweight="bold", y=0.98)
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)

# (1) Loss 곡선
ax1 = fig.add_subplot(gs[0, 0])
ax1.plot(history["train_loss"], label="Train", color="steelblue")
ax1.plot(history["val_loss"],   label="Val",   color="tomato")
ax1.set_title("Loss"); ax1.set_xlabel("Epoch"); ax1.legend(); ax1.grid(True, alpha=0.3)

# (2) Accuracy 곡선
ax2 = fig.add_subplot(gs[0, 1])
ax2.plot([a*100 for a in history["train_acc"]], label="Train", color="steelblue")
ax2.plot([a*100 for a in history["val_acc"]],   label="Val",   color="tomato")
ax2.axhline(test_acc*100, color="green", linestyle="--", label=f"Test {test_acc*100:.1f}%")
ax2.set_title("Accuracy (%)"); ax2.set_xlabel("Epoch"); ax2.legend(); ax2.grid(True, alpha=0.3)

# (3) Learning Rate (scheduler)
ax3 = fig.add_subplot(gs[0, 2])
lrs = [config["training"]["learning_rate"] * 0.5 * (1 + np.cos(np.pi * e / EPOCHS))
       for e in range(EPOCHS)]
ax3.plot(lrs, color="purple")
ax3.set_title("Learning Rate (Cosine)"); ax3.set_xlabel("Epoch"); ax3.grid(True, alpha=0.3)

# (4) Confusion Matrix
CLASS_NAMES = ["HC", "MCI", "Prodromal", "PD"]
used_labels = sorted(set(all_labels))
names = [CLASS_NAMES[i] if i < len(CLASS_NAMES) else str(i) for i in used_labels]
cm = confusion_matrix(all_labels, all_preds, labels=used_labels)
ax4 = fig.add_subplot(gs[1, 0])
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=names, yticklabels=names, ax=ax4)
ax4.set_title("Confusion Matrix"); ax4.set_xlabel("Predicted"); ax4.set_ylabel("True")

# (5) Per-class F1
ax5 = fig.add_subplot(gs[1, 1])
rep = classification_report(all_labels, all_preds, labels=used_labels,
                            target_names=names, output_dict=True, zero_division=0)
f1s  = [rep[n]["f1-score"] for n in names]
bars = ax5.bar(names, f1s, color=["steelblue","tomato","orange","green"][:len(names)])
for bar, v in zip(bars, f1s):
    ax5.text(bar.get_x() + bar.get_width()/2, v + 0.01, f"{v:.2f}",
             ha="center", fontsize=10)
ax5.set_ylim(0, 1.1); ax5.set_title("Per-class F1 Score"); ax5.grid(True, alpha=0.3, axis="y")

# (6) 요약 텍스트
ax6 = fig.add_subplot(gs[1, 2])
ax6.axis("off")
summary = (
    f"=== 학습 요약 ===

"
    f"총 에폭:      {EPOCHS}
"
    f"배치 크기:    {BS}
"
    f"학습률:       {config['training']['learning_rate']}

"
    f"Train Loss:   {history['train_loss'][-1]:.4f}
"
    f"Val   Loss:   {history['val_loss'][-1]:.4f}

"
    f"Train Acc:    {history['train_acc'][-1]*100:.1f}%
"
    f"Val   Acc:    {history['val_acc'][-1]*100:.1f}%
"
    f"Best Val Acc: {best_val_acc*100:.1f}%
"
    f"Test  Acc:    {test_acc*100:.1f}%

"
    f"Macro F1:     {rep['macro avg']['f1-score']:.4f}
"
    f"Weighted F1:  {rep['weighted avg']['f1-score']:.4f}"
)
ax6.text(0.05, 0.95, summary, transform=ax6.transAxes,
         fontsize=11, verticalalignment="top", fontfamily="monospace",
         bbox=dict(boxstyle="round", facecolor="lightyellow", alpha=0.8))

plt.savefig(os.path.join(SAVE_DIR, "training_report.png"), dpi=150, bbox_inches="tight")
plt.show()
print("
리포트 저장:", os.path.join(SAVE_DIR, "training_report.png"))
print("
분류 리포트:")
print(classification_report(all_labels, all_preds, labels=used_labels,
                             target_names=names, zero_division=0))
